In [1]:
from langchain_huggingface import HuggingFaceEmbeddings


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)   



c:\Users\Labdhi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2643.56it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
from langchain_chroma import Chroma

vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="db/chroma_db",  
)

In [3]:
import os
from dotenv import load_dotenv
load_dotenv() 
from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

In [4]:
def ReformulateQuery(user_query,chat_history):
    prompt=f""" Given a chat history and the latest user question which might reference context in the chat history, formulate a standalone question which can be understood without the chat history. Do NOT answer the question, just reformulate it if needed and otherwise return it as is.
    user query:{user_query}
    chat history:{chat_history}"""
    chat_completion = client.chat.completions.create(
            messages=[
                {"role": "user", "content": prompt}
            ],
            model="llama-3.3-70b-versatile",
        )
    return chat_completion.choices[0].message.content

In [5]:
chat_history={}
def history_aware_chatbot():
    cnt=1
    while True:
        user_query=input("Enter your query")
        if(user_query=='quit'):
            break
        reformulated_query=user_query
        if len(chat_history)!=0:
            reformulated_query=ReformulateQuery(user_query,chat_history)

        results=vector_store.similarity_search(reformulated_query,k=10)
        context_for_llm = "\n\n".join([doc.page_content for doc in results])
        prompt=f"""
        Answer the question based on the given context.
        context: {context_for_llm}
        question: {reformulated_query}
        """
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "user", "content": prompt}
            ],
            model="llama-3.3-70b-versatile",
        )
        chat_history[cnt]={'query':reformulated_query,'response':chat_completion.choices[0].message.content}
        cnt=cnt+1
        print(chat_completion.choices[0].message.content)



In [6]:
history_aware_chatbot()

Hilti is a Liechtenstein multinational company that develops, manufactures, and markets products for the construction, building maintenance, energy, and manufacturing industries. The company concentrates mainly on:

1. Anchoring systems
2. Fire protection systems
3. Installation systems
4. Measuring and detection tools (such as laser levels, range meters, and line lasers)
5. Power tools (such as hammer drills, demolition hammers, diamond drills, cordless electric drills, heavy angle drills, and power saws)
6. Related software and services.

These products are mainly sold to professional end-users.
The multinational company Hilti is based in Schaan, Liechtenstein.
The Hilti company, Maschinenbau Hilti OHG, was founded in 1941 by Martin and Eugen Hilti.
It seems like you want me to formulate a standalone question based on the given context. Here's a possible question:

What are the average scores of Computer Engineering and Electronics and Telecommunications, and what employee developmen

KeyboardInterrupt: Interrupted by user